In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os
import matplotlib.pyplot as plt
from sklearn.linear_model import ElasticNet
from sklearn.metrics import r2_score
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, average_precision_score,
    roc_auc_score, auc as compute_auc, r2_score
)


In [2]:
def load_heart_disease_data(filepath='data/heart_disease_uci.csv'):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")
    
    df = pd.read_csv(filepath)
    if df.empty:
        raise ValueError(f"CSV file is empty: {filepath}")
    
    return df



In [3]:
def preprocess_data(df):
    df = df.copy()
    df.replace("?", np.nan, inplace = True)
    df = df.apply(pd.to_numeric, errors = 'coerce')
    df =df.fillna(df.median(numeric_only=True))
    return df

In [4]:

def prepare_regression_data(df, target='chol'):
    df = df.dropna(subset =[target])
    X = df.drop(columns=[target])
    y = df[target]
    return X, y

In [5]:
def prepare_classification_data(df, target='num'):
    df = df.copy()
    
    df[target] = (df[target] > 0).astype(int)
    X =df.drop(columns=[target, 'chol'])
    y = df[target]
    return X,y

In [6]:
def split_and_scale(X, y, test_size=0.2, random_state=42):
    stratify = y if len(pd.Series(y).unique()) == 2 else None
    X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=test_size, random_state=random_state, stratify =stratify)
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, y_train, y_test, scaler

In [9]:
def train_elasticnet_grid(X_train, y_train, l1_ratios, alphas):
    results = []
    
    for l1 in l1_ratios:
        for alpha in alphas:
            model = ElasticNet (alpha=alpha, l1_ratio=l1, max_iter =5000, random_state = 42)
            model.fit(X_train, y_train)
            
            r2 = r2_score(y_train, model.predict(X_train))
            
            results.append({
                'l1_ratio':l1, 
                'alpha':alpha, 
                'r2_score':r2,
                'model':model
                })
           
    results_df = pd.DataFrame(results)
    return results_df

In [10]:
def create_r2_heatmap(results_df, l1_ratios, alphas, output_path=None):
    heatmap_data = results_df.pivot(index='alpha', columns ='l1_ratio', values ='r2_score')
    plt.figure(figsize = (6,4))
    sns.heatmap(heatmap_data, annot=True, cmap = 'viridis', fmt= '.3f', cbar_kws = {'label':'R2 Score'})
    plt.xlabel('L1 ratio')
    plt.ylabel('Alpha')
    plt.title('R2 Score Heatmap for ElasticNet Hyperparameters')
    if output_path:
        plt.savefig(output_path)
    fig = plt.gcf()
    return fig
    

In [11]:
def get_best_elasticnet_model(X_train, y_train, X_test, y_test, 
                               l1_ratios=None, alphas=None):
    if l1_ratios is None:
        l1_ratios = [0.3, 0.5, 0.7]
    if alphas is None:
        alphas = [0.01, 0.1, 1.0]
    results_df = train_elasticnet_grid(X_train, y_train, l1_ratios, alphas)
    
    test_r2_scores = []
    
    for model in results_df['model']:
        y_pred_test = model.predict(X_test)
        test_r2 = r2_score(y_test, y_pred_test)
        test_r2_scores.append(test_r2)
        
    results_df['test_r2']= test_r2_scores
    best_index = results_df['test_r2'].idxmax()
    best_row = results_df.loc['best_index']
        
    best_model = best_row['model']
    best_l1_ratios = best_row['l1_ratio']
    best_alphas = best_row['alpha']
        
    train_r2 = best_row['r2_score']
    test_r2 = best_row['test_r2']
        
    return {
            'model': best_model,
            'best_l1_ratio': best_l1_ratios,
            'best_alpha': best_alphas,
            'train_r2': train_r2,
            'test_r2': test_r2,
            'results_df': results_df
        }

        


In [12]:
def train_logistic_regression_grid(X_train, y_train, param_grid=None):
    if param_grid is None:
        param_grid = {
            'C': [0.001, 0.01, 0.1, 1, 10, 100],
            'penalty': ['l2'],
            'solver': ['lbfgs']
        }
        model = LogisticRegression(max_iter=1000)
    
    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,
        scoring="roc_auc"
    )
    grid.fit(X_train, y_train)
    return grid

In [13]:
def train_knn_grid(X_train, y_train, param_grid=None):
    if param_grid is None:
        param_grid = {
            'n_neighbors': [3, 5, 7, 9, 11, 15, 20],
            'weights': ['uniform', 'distance'],
            'metric': ['euclidean', 'manhattan']
        }
    if param_grid is None:
        param_grid ={
            'n_neighbors': [3,5,7,9,11,15,20],
            'weights': ['uniform', 'distance'],
            'metric': ['euclidean', 'manhattan']
        }
    model = KNeighborsClassifier()
    
    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,
        scoring="roc_auc"
    )
    grid.fit(X_train, y_train)
    return grid
       

In [14]:
def train_knn_grid(X_train, y_train, param_grid=None):
    if param_grid is None:
        param_grid = {
            'n_neighbors': [3, 5, 7, 9, 11, 15, 20],
            'weights': ['uniform', 'distance'],
            'metric': ['euclidean', 'manhattan']
        }
    if param_grid is None:
        param_grid ={
            'n_neighbors': [3,5,7,9,11,15,20],
            'weights': ['uniform', 'distance'],
            'metric': ['euclidean', 'manhattan']
        }
    model = KNeighborsClassifier()
    
    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,
        scoring="roc_auc"
    )
    grid.fit(X_train, y_train)
    return grid

In [15]:
def get_best_logistic_regression(X_train, y_train, X_test, y_test, param_grid=None):
    grid = train_logistic_regression_grid(X_train, y_train, param_grid)
    
    best_model = grid.best_estimator_
    best_params = grid.best_params_
    cv_results_df = pd.DataFrame(grid.cv_results_)
    
    return{
        'model':best_model,
        'best_params':best_params,
        'cv_results_df':cv_results_df
    }

In [16]:
def get_best_knn(X_train, y_train, X_test, y_test, param_grid=None):
    grid = train_knn_grid(X_train, y_train,param_grid)
    best_model = grid.best_estimator_
    best_params = grid.best_params_
    best_k = best_params['n_neighbors']
    cv_results_df = pd.DataFrame(grid.cv_results_)
    
    return{
        'model':best_model,
        'best_params':best_params,
        'best_k':best_k,
        'cv_results_df':cv_results_df
    }